In [1]:
import os 
import requests
import json 
import random
import time 

In [2]:
URL = "http://localhost:11434/api/generate"
MODEL = "llama3"

In [3]:
SAVE_PATH = r"C:\Datasets\processed\time_forge_ver_2_syn.jsonl"
TARGET_SAMPLES = 1000

In [4]:
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)

In [5]:
TOPICS = [
"ARIMA basics","ARIMA parameters","ARIMA model fitting","ARIMA forecasting",
"stationarity concept","checking stationarity","ADF test","KPSS test",
"differencing technique","first order differencing","seasonal differencing",
"ACF interpretation","PACF interpretation","ACF vs PACF",
"time series trend","trend detection","trend removal",
"seasonality concept","seasonality detection","seasonal decomposition",
"moving average smoothing","simple moving average","weighted moving average",
"exponential smoothing","Holt linear trend","Holt-Winters method",
"SARIMA model","SARIMA parameters","SARIMA vs ARIMA",
"time series decomposition","additive decomposition","multiplicative decomposition",
"lag features","lag analysis","autocorrelation",
"partial autocorrelation","white noise series","random walk",
"time series forecasting basics","forecast horizon","multi-step forecasting",
"rolling forecast","walk-forward validation","backtesting",
"time series visualization","plotting time series","trend plotting",
"anomaly detection","outlier detection","spike detection",
"time series scaling","normalization","standardization",
"missing values in time series","interpolation methods","forward fill",
"backward fill","time series resampling","upsampling","downsampling",
"datetime indexing","time series indexing","frequency conversion",
"feature engineering for time series","window features","rolling statistics",
"variance in time series","covariance","correlation",
"seasonal indices","cyclical patterns","irregular components",
"time series noise","denoising techniques","smoothing filters",
"Fourier transform in time series","frequency domain analysis",
"trend vs seasonality","level component","residual component",
"time series regression","linear regression for time series",
"multivariate time series","univariate vs multivariate",
"deep learning for time series","LSTM basics","RNN basics",
"temporal patterns","pattern recognition","sequence modeling",
"time series evaluation","MAE metric","RMSE metric",
"forecast accuracy","error analysis","model comparison",
"baseline forecasting","naive forecast","moving average forecast",
"exponential smoothing forecast","time series pipelines","model selection",
"hyperparameter tuning in ARIMA","grid search ARIMA","auto ARIMA"
]

In [12]:
def build_prompt(topic):
    return f"""
Return ONLY a valid JSON object. No explanation. No markdown. No backticks. No extra text.

The output MUST strictly follow this schema:
{"instruction": "<string>", "response": "<string>"}

Rules:
- Use double quotes ONLY
- Do NOT include trailing commas
- Do NOT include newlines outside the JSON
- Escape all newlines inside response using \\n
- Do NOT include ``` or the word python
- Do NOT include comments
- Response must be complete and not cut off
- Code (if present) must be valid Python

Task:
Generate ONE training example for a Time Series AI assistant.

Topic: {topic}

Choose ONE type:
1) Explanation → explain the topic clearly (2–4 lines)
2) Code → write Python code using statsmodels (ARIMA or relevant)
3) Hybrid → 1–2 line explanation + code

Constraints:
- Instruction must clearly mention the topic
- Response must directly answer instruction
- No repetition
- No unrelated text

"""

In [ ]:
def is_valid(sample):
    if not sample:
        return False

    if "instruction" not in sample or "response" not in sample:
        return False

    instr = sample["instruction"]
    resp = sample["response"]

    return (
        isinstance(instr, str) and
        isinstance(resp, str) and
        len(instr) > 10 and
        len(resp) > 20 and
        "..." not in resp
    )

In [15]:
def generate_sample(topic):
    try:
        res = requests.post(
            URL,
            json={
                "model": MODEL,
                "prompt": build_prompt(topic),
                "stream": False,
                "format": "json",
                "options": {
                    "temperature": 0.7,
                    "num_predict": 1200
                }
            },
            timeout=120
        )

        raw = res.json()["response"]
        cleaned = clean_response(raw)

        return json.loads(cleaned)

    except Exception as e:
        print(f"Error: {e}")
        return None


In [16]:
dataset = []
attempts = 0
while len(dataset) < TARGET_SAMPLES:
    topic = random.choice(TOPICS)
    attempts += 1

    print(f"[{len(dataset)+1}/{TARGET_SAMPLES}] {topic}...", end=" ")

    sample = generate_sample(topic)

    if is_valid(sample):
        dataset.append(sample)
        print(" saved")
    else:
        print(" rejected")

    # save every 10 samples
    if len(dataset) % 10 == 0:
        with open(SAVE_PATH, "w", encoding="utf-8") as f:
            json.dump(dataset, f, indent=2)

    time.sleep(1)

[1/1000] correlation... Error: Invalid format specifier ' "<string>", "response": "<string>"' for object of type 'str'
 rejected
[1/1000] denoising techniques... Error: Invalid format specifier ' "<string>", "response": "<string>"' for object of type 'str'
 rejected
[1/1000] model comparison... Error: Invalid format specifier ' "<string>", "response": "<string>"' for object of type 'str'
 rejected
[1/1000] frequency conversion... Error: Invalid format specifier ' "<string>", "response": "<string>"' for object of type 'str'
 rejected
[1/1000] first order differencing... Error: Invalid format specifier ' "<string>", "response": "<string>"' for object of type 'str'
 rejected
[1/1000] multivariate time series... Error: Invalid format specifier ' "<string>", "response": "<string>"' for object of type 'str'
 rejected
[1/1000] smoothing filters... Error: Invalid format specifier ' "<string>", "response": "<string>"' for object of type 'str'
 rejected
[1/1000] forecast accuracy... Error: Inval

KeyboardInterrupt: 

In [ ]:
with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
print(f"\DONE: {len(dataset)} samples saved")
print(f"Attempts: {attempts}")

\DONE: 600 samples saved
Attempts: 811
